# 01. 데이터 다운로드 및 준비

HF `datasets`의 **`ksang/steamreviews`** (Kaggle Steam Reviews 미러, `review_text` + `review_score` 1/-1)에서
리뷰를 받아 정제·분할해 `data/{train,val,test}.csv`로 저장한다.

- 앞부분만 자르면 특정 게임(app_id 순 정렬)에 편중되므로 **셔플 후 샘플링**한다.
- 정제/분할 로직은 `src.data.pipeline` 모듈을 사용한다.

In [1]:
import sys
from pathlib import Path

# 프로젝트 루트를 import 경로에 추가 (notebooks/ 하위에서 실행되므로)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from datasets import load_dataset
from src.config import DATA_DIR, RANDOM_SEED
from src.data.pipeline import clean_reviews, split_data

DATASET_ID = "ksang/steamreviews"
LIMIT = 20000

/Users/gomuseo/Desktop/Python/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = load_dataset(DATASET_ID, split="train")
print(f"전체 리뷰 수: {len(ds):,}")
sample = ds.shuffle(seed=RANDOM_SEED).select(range(LIMIT))
df = sample.to_pandas()
df.head(3)

Repo card metadata block was not found. Setting CardData to empty.


전체 리뷰 수: 6,417,106


,app_id,app_name,review_text,review_score,review_votes
0,239200,Amnesia: A Machine for Pigs,"Little game, such shame, much disappoint, not ...",-1,0
1,27940,Dead Horde,10/10 GOTY 2014 Pros: -Shoot dead people. -Blo...,-1,0
2,22100,Mount & Blade,Now I'm not saying I know how to play this gam...,1,1


In [3]:
clean = clean_reviews(df, "review_text", "review_score")
print(f"정제 후: {len(clean):,}건")
print(clean["label"].value_counts(normalize=True).rename({1: "긍정", 0: "부정"}))

정제 후: 16,653건
label
긍정    0.821233
부정    0.178767
Name: proportion, dtype: float64


In [4]:
tr, va, te = split_data(clean, seed=RANDOM_SEED)
DATA_DIR.mkdir(parents=True, exist_ok=True)
tr.to_csv(DATA_DIR / "train.csv", index=False)
va.to_csv(DATA_DIR / "val.csv", index=False)
te.to_csv(DATA_DIR / "test.csv", index=False)
print(f"train={len(tr)} val={len(va)} test={len(te)} → {DATA_DIR}")

train=11657 val=2498 test=2498 → /Users/gomuseo/Desktop/Python/review-check/data
